# Born-Probability Snapshot Histograms

This notebook loads the saved Born-probability snapshot caches for the clean `nshell=1`, `dw_truncation=true`, `perfect_correction` runs at `N16x30` and `N16x40`, then plots histogram grids that compare the domain-wall region against the non-domain-wall region.

For each system size, the histogram figure uses:
- columns = `N_A_lower`, `N_B_lower`, `one_minus_N_A_upper`, `one_minus_N_B_upper`
- rows = increasing snapshot cycle
- each histogram = all values at fixed snapshot, flattened over sample and masked unit cells
- domain-wall region = the two wall neighborhoods `x_dw ± 1`


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np

HIST_BINS = 40
HIST_RANGE = (0.0, 1.0)
DW_RADIUS = 0
DW_COLOR = '#C44E52'
NON_DW_COLOR = '#4C72B0'
OBSERVABLE_ORDER = [
    'N_A_lower',
    'N_B_lower',
    'one_minus_N_A_upper',
    'one_minus_N_B_upper',
]
OBSERVABLE_TITLES = {
    'N_A_lower': r'$\langle N_{A,\mathrm{lower}} \rangle$',
    'N_B_lower': r'$\langle N_{B,\mathrm{lower}} \rangle$',
    'one_minus_N_A_upper': r'$\langle 1 - N_{A,\mathrm{upper}} \rangle$',
    'one_minus_N_B_upper': r'$\langle 1 - N_{B,\mathrm{upper}} \rangle$',
}


In [ ]:
def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'REPO_POLICY.md').exists() and (candidate / 'src' / 'fgtn').exists():
            return candidate
    raise RuntimeError(f'Could not locate repo root from {start}')


def load_npz_payload(path: Path) -> dict:
    with np.load(path, allow_pickle=False) as data:
        payload = {key: data[key] for key in data.files}
    payload['metadata'] = json.loads(str(payload['metadata_json']))
    return payload


def reference_dw_locations(nx: int) -> list[int]:
    half = int(nx) // 2
    w = max(1, int(nx) // 4)
    return [int(half - w), int(half + w)]


def wall_neighborhood_mask(nx: int, ny: int, dw_loc: list[int], radius: int = 1) -> tuple[np.ndarray, np.ndarray]:
    dw_mask = np.zeros((int(nx), int(ny)), dtype=bool)
    for x_dw in dw_loc:
        for dx in range(-int(radius), int(radius) + 1):
            dw_mask[(int(x_dw) + dx) % int(nx), :] = True
    non_dw_mask = ~dw_mask
    return dw_mask, non_dw_mask


def masked_values(payload: dict, observable: str, snapshot_idx: int, region_mask: np.ndarray) -> np.ndarray:
    arr = np.asarray(payload[observable][:, int(snapshot_idx), :, :], dtype=np.float64)
    values = arr[:, region_mask]
    return values.reshape(-1)


def payload_summary(name: str, payload: dict, dw_loc: list[int], dw_mask: np.ndarray, non_dw_mask: np.ndarray) -> dict:
    dw_xcols = np.where(dw_mask.any(axis=1))[0].astype(int).tolist()
    non_dw_xcols = np.where(non_dw_mask.any(axis=1))[0].astype(int).tolist()
    return {
        'config': name,
        'cache_format': payload['metadata']['cache_format'],
        'snapshot_cycles': payload['snapshot_cycles'].tolist(),
        'dw_loc': [int(x) for x in dw_loc],
        'dw_x_columns': dw_xcols,
        'non_dw_x_columns': non_dw_xcols,
        'dw_unit_cells': int(dw_mask.sum()),
        'non_dw_unit_cells': int(non_dw_mask.sum()),
        'N_A_lower_shape': tuple(int(x) for x in payload['N_A_lower'].shape),
        'N_B_lower_shape': tuple(int(x) for x in payload['N_B_lower'].shape),
        'one_minus_N_A_upper_shape': tuple(int(x) for x in payload['one_minus_N_A_upper'].shape),
        'one_minus_N_B_upper_shape': tuple(int(x) for x in payload['one_minus_N_B_upper'].shape),
        'source_run_dir_relative': payload['metadata']['source_run_dir_relative'],
    }


def plot_histogram_grid(payload: dict, *, system_label: str, dw_radius: int = 1):
    nx = int(payload['metadata']['Nx'])
    ny = int(payload['metadata']['Ny'])
    dw_loc = reference_dw_locations(nx)
    dw_mask, non_dw_mask = wall_neighborhood_mask(nx, ny, dw_loc, radius=dw_radius)
    snapshot_cycles = np.asarray(payload['snapshot_cycles'], dtype=np.int64)
    nrows = int(snapshot_cycles.size)
    ncols = len(OBSERVABLE_ORDER)
    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(3.4 * ncols, 2.4 * nrows),
        sharex=True,
        constrained_layout=True,
    )
    axes = np.atleast_2d(axes)

    for row_idx, cycle in enumerate(snapshot_cycles):
        for col_idx, observable in enumerate(OBSERVABLE_ORDER):
            ax = axes[row_idx, col_idx]
            dw_values = masked_values(payload, observable, row_idx, dw_mask)
            non_dw_values = masked_values(payload, observable, row_idx, non_dw_mask)
            ax.hist(non_dw_values, bins=HIST_BINS, range=HIST_RANGE, color=NON_DW_COLOR, alpha=0.55, label='non-DW')
            ax.hist(dw_values, bins=HIST_BINS, range=HIST_RANGE, color=DW_COLOR, alpha=0.65, label='DW')
            ax.set_xlim(*HIST_RANGE)
            if row_idx == 0:
                ax.set_title(OBSERVABLE_TITLES[observable], fontsize=11)
                ax.legend(fontsize=8, frameon=False)
            if col_idx == 0:
                ax.set_ylabel(f'Cycle {int(cycle)}\nCount')
            if row_idx == nrows - 1:
                ax.set_xlabel('Born probability')

    fig.suptitle(
        f"{system_label} | nshell=1 | dw_truncation=True | DW region: x_dw ± {int(dw_radius)}",
        fontsize=14,
    )
    plt.show()


# Defect order and titles matching analysis.ipynb convention
DEFECT_ORDER = ['A_upper', 'A_lower', 'B_upper', 'B_lower']
DEFECT_TITLES = {
    'A_upper': 'A upper defect',
    'A_lower': 'A lower defect (1-occ)',
    'B_upper': 'B upper defect',
    'B_lower': 'B lower defect (1-occ)',
}


def _compute_defect_maps(payload: dict, sample_idx: int) -> dict[str, np.ndarray]:
    """Convert stored Born-probability arrays to defect densities.

    Defect = deviation from ideal (fully filled lower band, empty upper band):
      A_upper defect = occ_Ap           = 1 - one_minus_N_A_upper
      A_lower defect = 1 - occ_Am       = 1 - N_A_lower
      B_upper defect = occ_Bp           = 1 - one_minus_N_B_upper
      B_lower defect = 1 - occ_Bm       = 1 - N_B_lower

    Returns dict[defect_key] → array of shape (n_snapshots, nx, ny).
    """
    s = sample_idx
    return {
        'A_upper': 1.0 - np.asarray(payload['one_minus_N_A_upper'], dtype=np.float64)[s],
        'A_lower': 1.0 - np.asarray(payload['N_A_lower'],            dtype=np.float64)[s],
        'B_upper': 1.0 - np.asarray(payload['one_minus_N_B_upper'], dtype=np.float64)[s],
        'B_lower': 1.0 - np.asarray(payload['N_B_lower'],            dtype=np.float64)[s],
    }


def plot_heatmap_grid(payload: dict, *, system_label: str, sample_idx: int = 0):
    """Grid of spatially-resolved defect density heatmaps.

    Columns = defect type (A upper, A lower, B upper, B lower).
    Rows    = snapshot cycle (increasing downward).
    Color scale is fixed to [0, global_max] across all panels.
    """
    nx = int(payload['metadata']['Nx'])
    ny = int(payload['metadata']['Ny'])
    snapshot_cycles = np.asarray(payload['snapshot_cycles'], dtype=np.int64)
    nrows = int(snapshot_cycles.size)
    ncols = len(DEFECT_ORDER)

    defect_maps = _compute_defect_maps(payload, sample_idx)

    # Shared color range across all defect types and cycles
    vmin = 0.0
    vmax = float(max(defect_maps[k].max() for k in DEFECT_ORDER))

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(3.2 * ncols, 2.4 * nrows),
        constrained_layout=True,
    )
    axes = np.atleast_2d(axes)

    extent = [-0.5, nx - 0.5, -0.5, ny - 0.5]

    for row_idx, cycle in enumerate(snapshot_cycles):
        for col_idx, key in enumerate(DEFECT_ORDER):
            ax = axes[row_idx, col_idx]
            heatmap = defect_maps[key][row_idx].T  # (nx,ny) → (ny,nx) for imshow
            im = ax.imshow(
                heatmap,
                origin='lower',
                aspect='auto',
                extent=extent,
                vmin=vmin,
                vmax=vmax,
                cmap='plasma',
                interpolation='nearest',
            )
            if row_idx == 0:
                ax.set_title(DEFECT_TITLES[key], fontsize=11)
            if col_idx == 0:
                ax.set_ylabel(f'Cycle {int(cycle)}\ny')
            else:
                ax.set_ylabel('y')
            if row_idx == nrows - 1:
                ax.set_xlabel('x')
            fig.colorbar(im, ax=ax, shrink=0.85, pad=0.02)

    fig.suptitle(
        f"{system_label} | nshell=1 | dw_truncation=True"
        f" | defect density | sample idx={sample_idx}"
        f" | color range [{vmin:.3e}, {vmax:.3e}]",
        fontsize=12,
    )
    plt.show()


In [ ]:
import sys

REPO_ROOT = find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT / 'src'))

CACHE_ROOT = REPO_ROOT / 'notebooks' / 'born_probability_covariance_snapshots'
CACHE_FILES = {
    'N16x30': CACHE_ROOT / 'N16x30_nsh1_perfect_correction_born_rule_probabilities.npz',
    'N16x40': CACHE_ROOT / 'N16x40_nsh1_perfect_correction_born_rule_probabilities.npz',
}

results = {}
summaries = {}
for system_name, cache_path in CACHE_FILES.items():
    if not cache_path.exists():
        raise FileNotFoundError(f'Missing Born-probability cache: {cache_path}')
    payload = load_npz_payload(cache_path)
    nx = int(payload['metadata']['Nx'])
    ny = int(payload['metadata']['Ny'])
    dw_loc = reference_dw_locations(nx)
    dw_mask, non_dw_mask = wall_neighborhood_mask(nx, ny, dw_loc, radius=DW_RADIUS)
    results[system_name] = payload
    summaries[system_name] = payload_summary(system_name, payload, dw_loc, dw_mask, non_dw_mask)

print(json.dumps(summaries, indent=2))


In [ ]:
from fgtn.classA_U1FGTN import classA_U1FGTN


def _g_snapshots_path(payload: dict) -> Path:
    rel = payload['metadata']['source_run_dir_relative']
    return REPO_ROOT / rel / 'batch_00000_snapshots.npy'


def load_g_snapshots(payload: dict) -> np.ndarray:
    """Load G covariance snapshots; returns array (n_samples, n_snapshots, Nlayer, Nlayer)."""
    p = _g_snapshots_path(payload)
    if not p.exists():
        raise FileNotFoundError(f'G snapshots not found: {p}')
    return np.load(p, mmap_mode='r')


def build_model(payload: dict) -> classA_U1FGTN:
    """Instantiate classA_U1FGTN from snapshot metadata (DW=False skips heavy setup)."""
    m = payload['metadata']
    return classA_U1FGTN(
        int(m['Nx']),
        int(m['Ny']),
        DW=False,   # only Nx/Ny needed for local_chern_marker_flat
        nshell=None,
    )


def compute_chern_markers(payload: dict) -> np.ndarray:
    """Compute sample-averaged local Chern marker for each snapshot cycle.

    Returns array of shape (n_snapshots, Nx, Ny) with tanh applied.
    """
    G_snaps = load_g_snapshots(payload)       # (S, T, Nlayer, Nlayer)
    model = build_model(payload)
    S, T = G_snaps.shape[:2]
    Nx, Ny = model.Nx, model.Ny
    acc = np.zeros((T, Nx, Ny), dtype=np.float64)
    for s in range(S):
        for t in range(T):
            acc[t] += model.local_chern_marker_flat(
                np.asarray(G_snaps[s, t], dtype=np.complex128),
                apply_tanh=True,
            )
    return acc / S   # (T, Nx, Ny)


def plot_chern_marker_grid(payload: dict, *, system_label: str):
    """2-row × 2-col grid of sample-averaged local Chern marker heatmaps, one per snapshot cycle."""
    nx = int(payload['metadata']['Nx'])
    ny = int(payload['metadata']['Ny'])
    snapshot_cycles = np.asarray(payload['snapshot_cycles'], dtype=np.int64)
    dw_loc = reference_dw_locations(nx)

    print(f'Computing Chern markers for {system_label} ({len(snapshot_cycles)} snapshots × samples)…')
    C_avg = compute_chern_markers(payload)   # (T, Nx, Ny)

    nrows, ncols = 2, 2
    fig, axes = plt.subplots(nrows, ncols, figsize=(5.5 * ncols, 4.2 * nrows), constrained_layout=True)
    axes = axes.ravel()
    extent = [-0.5, nx - 0.5, -0.5, ny - 0.5]

    for idx, (ax, cycle) in enumerate(zip(axes, snapshot_cycles)):
        heatmap = C_avg[idx].T   # (ny, nx) for imshow
        im = ax.imshow(
            heatmap,
            origin='lower',
            aspect='auto',
            extent=extent,
            vmin=-1.0,
            vmax=1.0,
            cmap='RdBu_r',
            interpolation='nearest',
        )
        # Mark DW locations
        for xdw in dw_loc:
            ax.axvline(xdw, color='k', lw=1.0, ls='--', alpha=0.6)
        ax.set_title(f'Cycle {int(cycle)}', fontsize=11)
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        fig.colorbar(im, ax=ax, shrink=0.85, pad=0.02, label='tanh(C)')

    fig.suptitle(
        f"{system_label} | sample-averaged local Chern marker | DW at x={dw_loc}",
        fontsize=13,
    )
    plt.show()
    return C_avg


## N16x30


In [ ]:
plot_histogram_grid(results['N16x30'], system_label='N16x30', dw_radius=DW_RADIUS)


### N16x30 Defect Density Heatmaps

In [ ]:
plot_heatmap_grid(results['N16x30'], system_label='N16x30')

### N16x30 Local Chern Marker (sample-averaged)

In [ ]:
C_avg_N16x30 = plot_chern_marker_grid(results['N16x30'], system_label='N16x30')

## N16x40


In [ ]:
plot_histogram_grid(results['N16x40'], system_label='N16x40', dw_radius=DW_RADIUS)


### N16x40 Defect Density Heatmaps

In [ ]:
plot_heatmap_grid(results['N16x40'], system_label='N16x40')

### N16x40 Local Chern Marker (sample-averaged)

In [ ]:
C_avg_N16x40 = plot_chern_marker_grid(results['N16x40'], system_label='N16x40')